In [1]:
import pandas as pd
import xarray as xr
import numpy as np

In [ ]:
df_ais = pd.read_csv('../data/PositionReport_cleaned.csv')
df_ais['Time'] = pd.to_datetime(df_ais['Time'])
print(f"{len(df_ais)} GPS points have been loaded")

wind_ds = xr.open_mfdataset('wind_20*.nc', engine='h5netcdf')
waves_ds = xr.open_mfdataset('waves_20*.nc', engine='h5netcdf')
waves_extra_ds = xr.open_mfdataset('wave_ext_*.nc', engine='h5netcdf')
wind_extra_ds = xr.open_mfdataset('wind_extra_*.nc', engine='h5netcdf')
tides_ds = xr.open_mfdataset('tides_*.nc', engine='h5netcdf')

datasets = [wind_ds, waves_ds, waves_extra_ds, wind_extra_ds, tides_ds]
validated_datasets = []

for ds in datasets:
    if 'valid_time' in ds.dims:
        ds = ds.rename({'valid_time': 'time'})
    validated_datasets.append(ds)

wind_ds, waves_ds, waves_extra_ds, wind_extra_ds, tides_ds = validated_datasets


print("Interpolation test for the first AIS point:")
test_time = df_ais['Time'].iloc[0]
test_lat = df_ais['Lat'].iloc[0]
test_lon = df_ais['Lon'].iloc[0]

print(f"True ship position (AIS):")
print(f"Time: {test_time}")
print(f"Coordinates: latitude {test_lat:.4f}, longitude {test_lon:.4f}")

# where the algorithm found the closest point on the rigid weather map
nearest_grid = wind_ds.sel(time=test_time, latitude=test_lat, longitude=test_lon, method="nearest")
grid_time = pd.to_datetime(nearest_grid['time'].values)
grid_lat = nearest_grid['latitude'].values
grid_lon = nearest_grid['longitude'].values

print(f"Closest satellite node:")
print(f"Captured time: {grid_time}")
print(f"Captured coordinates: latitude {grid_lat:.4f}, longitude {grid_lon:.4f}")
print(f"Algorithm will fetch wind (u: {nearest_grid['u10'].values:.2f} m/s) from the satellite node")



# time and space interpolation for each dataset
# coordinates for interpolation of the ship (lon, lat, time)
t = xr.DataArray(df_ais['Time'].values, dims="points")
lat = xr.DataArray(df_ais['Lat'].values, dims="points")
lon = xr.DataArray(df_ais['Lon'].values, dims="points")

interp_wind = wind_ds.interp(time=t, latitude=lat, longitude=lon, method="nearest")
interp_waves = waves_ds.interp(time=t, latitude=lat, longitude=lon, method="nearest")
interp_waves_extra = waves_extra_ds.interp(time=t, latitude=lat, longitude=lon, method="nearest")
interp_wind_extra = wind_extra_ds.interp(time=t, latitude=lat, longitude=lon, method="nearest")
interp_tides = tides_ds.interp(time=t, latitude=lat, longitude=lon, method="nearest")


df_ais['u10'] = interp_wind['u10'].values
df_ais['v10'] = interp_wind['v10'].values
df_ais['H_s'] = interp_waves['swh'].values

for var in waves_extra_ds.data_vars:
    df_ais[var] = interp_waves_extra[var].values

for var in wind_extra_ds.data_vars:
    df_ais[var] = interp_wind_extra[var].values

for var in tides_ds.data_vars:
    df_ais[var] = interp_tides[var].values


# filling out gaps on the map edges for all satellite data, if any
sat_variables = ['u10', 'v10', 'H_s'] + list(waves_extra_ds.data_vars) + list(wind_extra_ds.data_vars) + list(tides_ds.data_vars)
for column in sat_variables:
    df_ais[column] = df_ais[column].ffill().bfill()

# true wind speed from vectors u, v
df_ais['True_Wind_Speed_ms'] = np.sqrt(df_ais['u10']**2 + df_ais['v10']**2)


file = 'MasterSet.csv'
df_ais.to_csv(file, index=False)

print(f"{file} has {len(df_ais)} rows: {len(df_ais.columns)}")

156942 GPS points have been loaded
Interpolation test for the first AIS point:
True ship position (AIS):
Time: 2023-02-02 00:06:00
Coordinates: latitude 56.8267, longitude -0.1283
Closest satellite node:
Captured time: 2023-02-02 00:00:00
Captured coordinates: latitude 56.7500, longitude -0.2500
Algorithm will fetch wind (u: 6.93 m/s) from the satellite node
MasterSet.csv has 156942 rows: 17
